# Correlating Demographic Features Among Credit Card Clients to Defaulting

## I. Project Proposal

1.	Introduction

    Moneylending is a risky business. Nonetheless, profitable businesses pull it off every day. They do so in part by utilizing large data sets containing information about credit clients around the world. These data sets can be used to train machine learning (ML) models capable of predicting a potential client’s creditworthiness. The current proposal seeks to examine one of these data sets using ML and to explore how different demographics may or may not contribute to an individual’s likelihood of defaulting. A unique opportunity for this study arose due to a very particular data set. In the early 2000’s, Taiwan experienced a “credit crisis” of sorts because banks were approving credit cards for individuals regardless of their financial history in a misguided effort to gain more business. Looking back on it, this was clearly a huge mistake, but thankfully they recorded the data, and it is available now for the current analysis. The data set is unique because it tracks a large sample of customers from a wide range of demographics along with information about if and when they defaulted. Thus, the present study will examine these demographics and correlate them with defaulting.
2.	Data Set
    
    This data set is part of an entry on UCI’s Machine Learning Repository entitled “default of credit card clients Data Set.” The data set is publicly available at this link. There is also a public notebook on Kaggle where the user took the raw data set and applied a few basic machine learning models here. The presently proposed methods are expected to differ significantly from this original analysis, mainly in that there was no pre-processing of the data in the Kaggle notebook. Additionally, the main question of the original analysis is not clearly defined, and the methods seem to be focused on predicting creditworthiness instead of an analysis of socioeconomic factors which may or may not contribute to the likelihood of defaulting. The factors included in the dataset are initially offered credit limit, gender, education, marital status, age, history of past payment, amount of bill statement, and amount of previous payment. The final three factors are split into six columns each for a time period of six months of data collection, for a grand total of 23 factors. The dependent variable is one final column containing a yes/no answer to the question “did this customer default?”
3.	Objective
    
    The main question of the proposed analysis is what effect do demographic factors have on a person’s circumstance of defaulting on a credit card, if any? This is not to be confused with predicting a client’s creditworthiness, leave that to the professionals. Instead, this study is meant to be a sociological analysis of human conditions and their potential relationship with financial decisions. It is predicted that the following categories will be less likely to default: female, well-educated, married, age above 30, high initial offering of credit, good payment history, low bill statements, and previous payment close to previous bill statement.
4.	Machine Learning Models
    
    First, before any models are applied, the data needs to be pre-processed. One problem is that the data are imbalanced. It is 60% female and 40% male, among other imbalances. An initial filtering will be completed to ensure that the demographics of gender, education, and marital status at a minimum are balanced. Depending on the feasibility, other factors will also be balanced, but there are only 30,000 data points, and the initial filtering may decrease this significantly. If these demographics are not balanced, the conclusions may be inaccurate by overestimating probabilities for the majority in each category and vice versa for the minority.
    
    Once preprocessing is complete, data analysis will begin with a multiple linear regression model. Many of the categories are numeric, and this method is expected to do well in predicting the effects of credit usage. However, there are a number of categorical factors as well, so a second analysis will be completed with a logistic regression model. Finally, the regression models will be compared with a cluster model because clusters are likely to form with sociological data sets like these.
5.	Probability of Success
    
    This project is expected to have a high probability of success because the data set is relatively clean and contains a sufficient number of samples to generate robust models with large training sets. It should be straightforward to then use these models to answer the question posited in the objective section.


## II. Methodology and Data Preprocessing

To preface, some data had to be removed from the raw data because they had empty cells where education and marital status should have been. There is nothing in the data set documentation indicating what this means, possibly the clients refused to give this information out or it was unknown to the bank. Thankfully this only resulted in the loss of around 850 data points out of the initial 30000.

Now to preprocessing. First, the data will be loaded into a data frame and one-hot encoded to utilize the Pandas .describe() method to analyze how the *demographic* data are currently distributed prior to any analysis. Emphasis on the demographic data because these are the ones needing to be balanced. The ultimate goal is to balance the data on the demographic factors of sex, education, marital status, and age as best as possible.

In [2]:
import pandas as pd

#Load credit client data into a Pandas dataframe and display result.
credit = pd.read_csv('data/credit.csv')
credit

,sex,education,marital,age,limit_bal,pay1,pay2,pay3,pay4,pay5,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,delinquent
0,F,Graduate school,Married,45,180000,0,0,0,2,0,...,24262,24782,26208,2690,5100,0,1058,2000,3000,N
1,F,University,Married,26,50000,0,0,0,0,0,...,49776,50937,49956,2200,2100,2000,2000,2000,2300,N
2,F,Graduate school,Single,34,430000,-1,-1,-1,2,-1,...,7370,4423,2544,1662,15783,52,4445,2556,3321,N
3,F,Graduate school,Married,35,470000,0,0,0,0,0,...,232961,217802,202092,9500,9000,8086,14700,7000,9000,N
4,F,University,Married,42,100000,-2,-2,-2,-2,-2,...,5524,4973,1000,1055,8090,5549,4973,1000,1000,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29158,M,High School,Married,48,50000,-1,-1,0,0,0,...,26832,27395,27940,33895,1439,959,994,1000,1013,N
29159,M,High School,Married,51,50000,1,2,0,0,0,...,28478,29074,29671,1,2213,1019,1055,1080,1245,Y
29160,M,University,Married,36,80000,-1,-1,0,0,0,...,26569,20082,25535,31091,20000,5000,10000,10000,0,N
29161,M,Graduate school,Married,44,360000,0,0,0,0,0,...,344545,140386,144042,15047,15000,14000,5008,10055,30015,N


In [3]:
from sklearn.preprocessing import OneHotEncoder

#Assign categorical data types
credit['sex']=pd.Categorical(credit['sex'])
credit['education']=pd.Categorical(credit['education'])
credit['marital']=pd.Categorical(credit['marital'])

#Initialize and fit OHE
enc = OneHotEncoder()
enc.fit(credit.select_dtypes('category'))

#Separate out categorical data, one hot encode, and describe
credit_numeric = credit.select_dtypes(exclude='category')
credit_feature_names = enc.get_feature_names_out()
credit_onehot = pd.DataFrame(data=enc.transform(credit.select_dtypes('category')).toarray(),columns=credit_feature_names)
credit_onehot=credit_onehot.set_index(credit.index)
credit_all = pd.concat([credit_onehot,credit_numeric],axis=1)

#Create separate data frame for demographic information
credit_demo = pd.concat([credit_onehot,credit['age']],axis=1)

#Describe demographic data frame
credit_demo.describe()

,sex_F,sex_M,education_Graduate school,education_High School,education_University,marital_Married,marital_Single,age
count,29163.000000,29163.000000,29163.000000,29163.000000,29163.000000,29163.000000,29163.000000,29163.000000
mean,0.603093,0.396907,0.361108,0.163563,0.475328,0.460344,0.539656,35.390563
std,0.489265,0.489265,0.480330,0.369885,0.499399,0.498433,0.498433,9.181333
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,28.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,34.000000
75%,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,41.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,79.000000


A comparison with Taiwan's demographic data from the same time period of this data set shows the only factor here which does not approximately reflect the population at large is sex. The median age in Taiwan in 2005 was 34.2, the marriage rate was 62% (averaged between 10 age groups for men and women), 79.2% went to college, and there are about 0.98 males for every 1 female in Taiwan. The marriage rate in this data set is a little bit low and the higher education rate a bit high, but still both are quite close to the overall population. Perhaps these values will change slightly when the sexes are balanced. Demographic data sourced from United Nations Department of Economic and Social Affairs website.

Next, to balance the sexes, a random sample of female clients will be extracted which is the same size as the number of male clients. Or, a random sample will be removed from the female clients until the two are balanced. Either way, the end goal is about a 50/50 split between male and female clients.

In [4]:
#Check number of male and female clients
credit['sex'].value_counts()

F    17588
M    11575
Name: sex, dtype: int64

At this point I realized I could use the RAND() function in Excel to randomize the order of data then group by sex to eliminate the need for sampling here, so the below code just removes the first 6013 female clients.

In [5]:
#Subtract 6013 female clients from the overall data frame to balance the sexes
credit=credit.iloc[6013:29163,:]
credit['sex'].value_counts()

F    11575
M    11575
Name: sex, dtype: int64

In [6]:
#Re-analyze one hot encoded demographics
#Initialize and fit OHE
enc = OneHotEncoder()
enc.fit(credit.select_dtypes('category'))

#Separate out categorical data, one hot encode, and describe
credit_numeric = credit.select_dtypes(exclude='category')
credit_feature_names = enc.get_feature_names_out()
credit_onehot = pd.DataFrame(data=enc.transform(credit.select_dtypes('category')).toarray(),columns=credit_feature_names)
credit_onehot=credit_onehot.set_index(credit.index)
credit_all = pd.concat([credit_onehot,credit_numeric],axis=1)

#Create separate data frame for demographic information
credit_demo = pd.concat([credit_onehot,credit['age']],axis=1)

#Describe demographic data frame
credit_demo.describe()

,sex_F,sex_M,education_Graduate school,education_High School,education_University,marital_Married,marital_Single,age
count,23150.000000,23150.000000,23150.000000,23150.000000,23150.000000,23150.000000,23150.000000,23150.000000
mean,0.500000,0.500000,0.364104,0.163542,0.472354,0.456847,0.543153,35.555810
std,0.500011,0.500011,0.481188,0.369868,0.499246,0.498145,0.498145,9.221597
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,28.000000
50%,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,1.000000,34.000000
75%,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,41.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,79.000000


Looks great, all other variables remained nearly the same and the sex demographic is now more representative of Taiwan's 2005 population.

## III. Results and Discussion - Regression Analyses

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import r2_score

#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'sex_F':'pay_amt6']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Fit linear model and score training and testing sets
reg = linear_model.LinearRegression()
reg.fit(x_train, y_train)
r2_train = reg.score(x_train,y_train)
r2_test = reg.score(x_test,y_test)
print('Training set r2: ',r2_train)
print('Testing set r2: ',r2_test)

Training set r2:  0.12873081963103727
Testing set r2:  0.11862025585378855


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing

#Scale data to avoid warnings and for better logistic regression performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#Try again with Logistic Regression
lr = LogisticRegression()
lr.fit(x_train_scaled,y_train.values.ravel())
r2_trainlr = lr.score(x_train_scaled,y_train)
r2_testlr = lr.score(x_test_scaled,y_test)
print('Training set Logistic Regression r2: ',r2_trainlr)
print('Testing set Logistic Regression r2: ',r2_testlr)

Training set Logistic Regression r2:  0.8077213822894168
Testing set Logistic Regression r2:  0.8075593952483802


In [9]:
from sklearn import metrics

#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=lr.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3496,  110],
       [ 781,  243]], dtype=int64)

I learned about the confusion matrix online and thought it could be useful here. It looks like this model is somewhat decent at detecting when a client will not default, but not so good at correctly predicting when a client will default.

In [10]:
#Compute and print logistic regression scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.8075593952483802
Precision: 0.6883852691218131
Recall: 0.2373046875


Here, I use some further scores to evaluate the logistic regression model. The one I am interested in is Recall, which means that in a given test set, if the set contains clients who default, the model can accurately predict it 23% of the time, which is not great.

Overall, an okay result for the linear regression. Logistic regression has a high score, but may be overfitting on a data set with so many factors like this one. Will now examine how each factor correlates with defaulting.

In [11]:
#Make sure delinquency is in its numeric form, calculate correlation matrix, display result for target column
credit_all['delinquent']=y.values
corrs = credit_all.corr()
print(corrs['delinquent'])

sex_F                       -0.044815
sex_M                        0.044815
education_Graduate school   -0.054785
education_High School        0.028660
education_University         0.031571
marital_Married              0.030530
marital_Single              -0.030530
age                          0.008084
limit_bal                   -0.155281
pay1                         0.328453
pay2                         0.267699
pay3                         0.242686
pay4                         0.218841
pay5                         0.207144
pay6                         0.189655
bill_amt1                   -0.019423
bill_amt2                   -0.014231
bill_amt3                   -0.014821
bill_amt4                   -0.010037
bill_amt5                   -0.004626
bill_amt6                   -0.003756
pay_amt1                    -0.075978
pay_amt2                    -0.057803
pay_amt3                    -0.054644
pay_amt4                    -0.050586
pay_amt5                    -0.057602
pay_amt6    

It's starting to look like demographic factors do not correlate with defaulting. To test this, I have created two models (one linear regression and one logistic regression) for a data set which only contains demographics and two models for another data set which only contains credit card usage metrics. First, the demographic models:

In [12]:
#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'sex_F':'age']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Fit linear model and score training and testing sets
reg = linear_model.LinearRegression()
reg.fit(x_train, y_train)
r2_train = reg.score(x_train,y_train)
r2_test = reg.score(x_test,y_test)
print('Training set r2: ',r2_train)
print('Testing set r2: ',r2_test)

Training set r2:  0.005918816405662275
Testing set r2:  0.006113619869960507


In [13]:
#Scale data to avoid warnings and for better logistic regression performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#Try again with Logistic Regression
lr = LogisticRegression()
lr.fit(x_train_scaled,y_train.values.ravel())
r2_trainlr = lr.score(x_train_scaled,y_train)
r2_testlr = lr.score(x_test_scaled,y_test)
print('Training set Logistic Regression r2: ',r2_trainlr)
print('Testing set Logistic Regression r2: ',r2_testlr)

Training set Logistic Regression r2:  0.7744600431965443
Testing set Logistic Regression r2:  0.7788336933045357


In [14]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=lr.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3606,    0],
       [1024,    0]], dtype=int64)

In [15]:
#Compute and print logistic regression scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.7788336933045357
Precision: 0.0
Recall: 0.0


C:\Users\Sam\anaconda3\envs\CSE6194\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Now it's really looking like demographics do not correlate with defaulting. The linear regression model has a near-zero score and the logistic regression model cannot make very good predictions at all. Next up, credit usage models:

In [16]:
#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'limit_bal':'pay6']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Fit linear model and score training and testing sets
reg = linear_model.LinearRegression()
reg.fit(x_train, y_train)
r2_train = reg.score(x_train,y_train)
r2_test = reg.score(x_test,y_test)
print('Training set r2: ',r2_train)
print('Testing set r2: ',r2_test)

Training set r2:  0.11712764246634222
Testing set r2:  0.11297936627428429


In [17]:
#Scale data to avoid warnings and for better logistic regression performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#Try again with Logistic Regression
lr = LogisticRegression()
lr.fit(x_train_scaled,y_train.values.ravel())
r2_trainlr = lr.score(x_train_scaled,y_train)
r2_testlr = lr.score(x_test_scaled,y_test)
print('Training set Logistic Regression r2: ',r2_trainlr)
print('Testing set Logistic Regression r2: ',r2_testlr)

Training set Logistic Regression r2:  0.8092332613390929
Testing set Logistic Regression r2:  0.8056155507559395


In [18]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=lr.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3493,  113],
       [ 787,  237]], dtype=int64)

In [19]:
#Compute and print logistic regression scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.8056155507559395
Precision: 0.6771428571428572
Recall: 0.2314453125


It seems like nearly all of the predictive power of these models (even though they only weakly predict) comes from each client's credit usage over time. This concludes the regression analysis.

## IV. Results and Discussion - Clustering Analysis

The data set wil now be examined with a KNN classifier. I am hopeful this will perform better since it is often said that sociological data tends to form clusters.

In [20]:
from sklearn.neighbors import KNeighborsClassifier

#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'sex_F':'pay_amt6']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Scale data to avoid warnings and for better KNN performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#KNN 3 neighbor model
neighbor3 = KNeighborsClassifier(n_neighbors=3)
neighbor3.fit(x_train_scaled,y_train.values.ravel())

#Compute and print scores for training set and testing set
print('Train Set KNN Score: ',neighbor3.score(x_train_scaled,y_train),'\nTest Set KNN Score: ',neighbor3.score(x_test_scaled,y_test))

Train Set KNN Score:  0.8629049676025918 
Test Set KNN Score:  0.7725701943844493


In [21]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=neighbor3.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3228,  378],
       [ 675,  349]], dtype=int64)

In [22]:
#Compute and print KNN scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.7725701943844493
Precision: 0.48005502063273725
Recall: 0.3408203125


KNN actually seems to do significantly better than logistic regression. Will now try again but with 5 nearest neighbors.

In [23]:
#KNN 5 neighbor model
neighbor5 = KNeighborsClassifier(n_neighbors=5)
neighbor5.fit(x_train_scaled,y_train.values.ravel())

#Compute and print scores for training set and testing set
print('Train Set KNN Score: ',neighbor5.score(x_train_scaled,y_train),'\nTest Set KNN Score: ',neighbor5.score(x_test_scaled,y_test))

Train Set KNN Score:  0.8409827213822895 
Test Set KNN Score:  0.790280777537797


In [24]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=neighbor5.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3314,  292],
       [ 679,  345]], dtype=int64)

In [25]:
#Compute and print KNN scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.790280777537797
Precision: 0.5416012558869702
Recall: 0.3369140625


No significant change with more neighbors. Try now with the modified data sets as before.

In [26]:
#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'sex_F':'age']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Scale data to avoid warnings and for better KNN performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#KNN 3 neighbor model
neighbor3 = KNeighborsClassifier(n_neighbors=3)
neighbor3.fit(x_train_scaled,y_train.values.ravel())

#Compute and print scores for training set and testing set
print('Train Set KNN Score: ',neighbor3.score(x_train_scaled,y_train),'\nTest Set KNN Score: ',neighbor3.score(x_test_scaled,y_test))

Train Set KNN Score:  0.7023218142548596 
Test Set KNN Score:  0.6997840172786177


In [27]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=neighbor3.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3058,  548],
       [ 842,  182]], dtype=int64)

In [28]:
#Compute and print KNN scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.6997840172786177
Precision: 0.2493150684931507
Recall: 0.177734375


Interesting, it looks like the KNN classifier is able to do slightly better with demographic factors than regression. Now for the credit history factors.

In [29]:
#Assign x and y and turn target into 0 for No and 1 for Yes
y_raw=credit[['delinquent']]
y=y_raw.eq('Y').mul(1)
x=credit_all.loc[:,'limit_bal':'pay6']

#Train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

#Scale data to avoid warnings and for better KNN performance
scaler = preprocessing.StandardScaler().fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#KNN 3 neighbor model
neighbor3 = KNeighborsClassifier(n_neighbors=3)
neighbor3.fit(x_train_scaled,y_train.values.ravel())

#Compute and print scores for training set and testing set
print('Train Set KNN Score: ',neighbor3.score(x_train_scaled,y_train),'\nTest Set KNN Score: ',neighbor3.score(x_test_scaled,y_test))

Train Set KNN Score:  0.8177105831533478 
Test Set KNN Score:  0.7732181425485961


In [30]:
#Generate classification predictions for the test set and calculate resulting confusion matrix
y_pred=neighbor3.predict(x_test_scaled)
metrics.confusion_matrix(y_test,y_pred)

array([[3174,  432],
       [ 618,  406]], dtype=int64)

In [31]:
#Compute and print KNN scores
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))
print("Precision:",metrics.precision_score(y_test, y_pred))
print("Recall:",metrics.recall_score(y_test, y_pred))

Accuracy: 0.7732181425485961
Precision: 0.48448687350835323
Recall: 0.396484375


Best recall value so far. While KNN can have some predictive power with demographic info, it still seems credit usage is a stronger predictor, though still a weak one as far as predictors in general go.

## V. Conclusions

After extensive analysis of this data set using multiple machine learning models, the following conclusions have been reached. Note that the conclusions technically apply to the population of Taiwan in 2005. A separate study would be needed to confirm that these conclusions apply to the general population, but one can infer that many modern societies will utilize credit cards in similar ways given similarities among global economies and banking systems.

1. All models in general have a difficult time making any accurate predictions when utilizing demographic factors alone.

2. Credit usage and payment history has a more significant effect on whether a client will default than their demographics.

3. While it is possible to reach accuracies near 80% with the right models, this value may be an artifact of over-fitting, as the recall scores for most models is quite low.

The answer to the question posed in the proposal, what effect do demographic factors have on a person’s circumstance of defaulting on a credit card, seems to be that there is little to no effect of demographics. Instead, a person's credit usage is more likely to determine whether they will default, regardless of demographics.